> # 🚀 Day 3 — RAG, Evaluation & Reliability
## 🤖 Customer Support AI Agent

Today we connect our Customer Support Agent to company knowledge 📖.

Instead of answering from generic LLM training data, our agent will retrieve factual policy and FAQ documents first and answer only from verified context 🛡️.

## 🏗️ 0. What Are We Building?

```text
       👤 Customer Question ("كم رسوم الشحن السريع؟")
                             ↓
              🔍 Semantic Retriever (ChromaDB)
                             ↓
     📄 Relevant Chunks + Metadata (Policy & FAQs)
                             ↓
              🧠 LLM (Grounded Generation)
                             ↓
      ✅ Factual, Verifiable Answer with Citations
```

## 📚 1. The Knowledge Base

For this challenge we use two rich knowledge sources:

- 📄 `company_policies.md`: Policy document covering shipping, returns, refunds, payments, and BNPL for GCC customers 🇸🇦 🇦🇪 🇰🇼.
- ❓ `faqs.json`: 15 structured customer questions and answers with metadata (category, country, update date).

In [ ]:
%pip install -q -U langchain langchain-openai langchain-chroma chromadb ragas pandas scikit-learn


In [1]:
from pathlib import Path
import json
import os
import re
import pandas as pd

# 🔍 Robust path resolution for local, IDE, notebook, or Colab environments
CWD = Path.cwd()
possible_dirs = [
    CWD / "data",
    CWD.parent / "data",
    CWD / "notebook" / "data",
    Path("/mnt/data"),
]
DATA_DIR = next((d for d in possible_dirs if d.exists()), CWD / "data")

POLICY_PATH = DATA_DIR / "company_policies.md"
FAQ_PATH = DATA_DIR / "faqs.json"
EVAL_PATH = DATA_DIR / "test_eval_queries.json"

print(f"📁 Using Data Directory: {DATA_DIR.resolve()}")
print("📄 Policy exists:", POLICY_PATH.exists())
print("❓ FAQs exist:", FAQ_PATH.exists())
print("🧪 Evaluation set exists:", EVAL_PATH.exists())


📁 Using Data Directory: D:\course\practs for course\data
📄 Policy exists: True
❓ FAQs exist: True
🧪 Evaluation set exists: True


In [2]:
policy_text = POLICY_PATH.read_text(encoding="utf-8")

with open(FAQ_PATH, "r", encoding="utf-8") as f:
    faqs = json.load(f)

with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_queries = json.load(f)

print("Policy characters:", len(policy_text))
print("FAQ records:", len(faqs))
print("Evaluation queries:", len(eval_queries))


Policy characters: 3767
FAQ records: 15
Evaluation queries: 20


## ✂️ 2. Why Chunking Matters

A vector database does not need the whole policy document as one large piece 📜.

We want smaller, modular pieces that contain **one coherent concept**:

```text
Large Policy Document
         ↓
  Markdown Header Split (#, ##, ###)
         ↓
  Recursive Character Splitter (500–800 tokens, 10–15% overlap)
         ↓
  Focused Chunks: Shipping Fees | Returns | Tamara/Tabby | Cancellation
```

This ensures the retriever returns precise information without polluting the LLM's context window 🎯.

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter # type: ignore

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,
)

policy_docs = header_splitter.split_text(policy_text)

print("Sections after heading split:", len(policy_docs))
print()
print(policy_docs[0].page_content[:500])
print(policy_docs[0].metadata)


Sections after heading split: 11

# 🏬 Gulf E-Commerce Customer Service Policies (GCC)  
## 🚚 1. Shipping & Delivery Policy  
### 📦 1.1 Domestic Shipping (Saudi Arabia & UAE)
- **Standard Shipping (الشحن العادي)**:
- **Delivery Time**: 3 to 5 business days across all major cities (Riyadh, Jeddah, Dammam, Dubai, Abu Dhabi).
- **Shipping Fees**: 25 SAR / 25 AED for orders below 200 SAR / AED.
- **Free Shipping**: Orders of 200 SAR / 200 AED and above qualify for free standard shipping.
- **Express Shipping (الشحن السريع)**:
- **Del
{'h1': '🏬 Gulf E-Commerce Customer Service Policies (GCC)', 'h2': '🚚 1. Shipping & Delivery Policy', 'h3': '📦 1.1 Domestic Shipping (Saudi Arabia & UAE)'}


In [4]:
# Split large sections into smaller retrieval chunks.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=300,
    separators=["\n\n", "\n", " ", ""],
)

policy_chunks = text_splitter.split_documents(policy_docs)

print("Final policy chunks:", len(policy_chunks))

for i, doc in enumerate(policy_chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:400])
    print("Metadata:", doc.metadata)


Final policy chunks: 11

--- Chunk 0 ---
# 🏬 Gulf E-Commerce Customer Service Policies (GCC)  
## 🚚 1. Shipping & Delivery Policy  
### 📦 1.1 Domestic Shipping (Saudi Arabia & UAE)
- **Standard Shipping (الشحن العادي)**:
- **Delivery Time**: 3 to 5 business days across all major cities (Riyadh, Jeddah, Dammam, Dubai, Abu Dhabi).
- **Shipping Fees**: 25 SAR / 25 AED for orders below 200 SAR / AED.
- **Free Shipping**: Orders of 200 SAR / 
Metadata: {'h1': '🏬 Gulf E-Commerce Customer Service Policies (GCC)', 'h2': '🚚 1. Shipping & Delivery Policy', 'h3': '📦 1.1 Domestic Shipping (Saudi Arabia & UAE)'}

--- Chunk 1 ---
### 🌍 1.2 Regional GCC Shipping (Kuwait, Bahrain, Qatar, Oman)
- **Delivery Time**: 4 to 7 business days via regional couriers (Aramex, DHL).
- **Shipping Fees**: Flat fee of 50 SAR / AED (or equivalent local currency).
- **Customs & Duties**: Handled and included in final checkout total; no additional payment upon delivery.  
---
Metadata: {'h1': '🏬 Gulf E-Commerce Custome

## 📋 3. Add FAQ Documents with Metadata

FAQs are already structured into clean Q&A units 💡.

Each FAQ becomes one document with rich metadata:
- `source_id`: e.g. `FAQ-001`
- `category`: e.g. `shipping`, `returns`, `bnpl`
- `target_country`: `KSA`, `UAE`, `GCC`
- `updated_at`: `2026-08-01`

Metadata filtering enables accurate multi-region and category-specific routing 🏷️.

In [5]:
faq_docs = []

for faq in faqs:
    metadata_dict = faq.get("metadata", {})
    target_country = faq.get("country") or metadata_dict.get("target_country", "GCC")
    updated_at = faq.get("updated_at") or metadata_dict.get("updated_at", "2026-08-01")
    
    faq_docs.append(
        Document(
            page_content=(
                f"Question: {faq['question']}\n"
                f"Answer: {faq['answer']}\n"
                f"Keywords: {', '.join(faq['keywords'])}"
            ),
            metadata={
                "source_id": faq["id"],
                "source_type": "faq",
                "category": faq["category"],
                "target_country": target_country,
                "updated_at": updated_at,
            },
        )
    )

for doc in faq_docs[:2]:
    print(doc.page_content)
    print(doc.metadata)
    print()


Question: كم رسوم الشحن السريع في السعودية؟
Answer: رسوم الشحن السريع داخل المملكة العربية السعودية هي 45 ريال سعودي، ويستغرق التوصيل من يوم إلى يومين عمل في المدن الرئيسية مثل الرياض وجدة.
Keywords: رسوم, الشحن السريع, السعودية, 45 ريال, يومين
{'source_id': 'FAQ-001', 'source_type': 'faq', 'category': 'shipping', 'target_country': 'KSA', 'updated_at': '2026-08-01'}

Question: كم يستغرق التوصيل إلى مدينة الرياض؟
Answer: التوصيل العادي إلى الرياض يستغرق من 3 إلى 5 أيام عمل، بينما التوصيل السريع يستغرق من يوم إلى يومين عمل فقط.
Keywords: التوصيل, الرياض, مدة, أيام, شحن
{'source_id': 'FAQ-002', 'source_type': 'faq', 'category': 'shipping', 'target_country': 'KSA', 'updated_at': '2026-08-01'}



## 🏷️ 4. Add Metadata to Policy Chunks

Metadata helps us trace where a retrieved chunk came from and verify citations:

- `source_type`: `policy` vs `faq`
- `region`: `GCC`
- `version`: `1.0.0`
- `updated_at`: `2026-08-01`
- Section hierarchy tags (`h1`, `h2`, `h3`)

In [6]:
for doc in policy_chunks:
    doc.metadata.update({
        "source_id": "COMPANY_POLICY",
        "source_type": "policy",
        "region": "GCC",
        "version": "1.0.0",
        "updated_at": "2026-08-01",
    })

all_docs = policy_chunks + faq_docs

print("Total documents for indexing:", len(all_docs))


Total documents for indexing: 26


## 🔢 5. Create Embeddings & ChromaDB Vector Store

An **Embedding** converts text into dense vector coordinates in semantic space 🌐. Similar meanings produce vectors that are close to each other.

```text
  Customer Question  ──►  Embedding (OpenAI text-embedding-3-small)
                                      ↓
  ChromaDB Vector Store  ◄──►  Cosine / Similarity Search
                                      ↓
  Top-k Relevant Chunks Returned
```

In [7]:
# Set your API key before running the embedding cells.
# Windows PowerShell:
# $env:OPENAI_API_KEY="your-key"
#
# Or set it in your environment before starting Jupyter.

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is not set.")
else:
    print("OPENAI_API_KEY is available.")


OPENAI_API_KEY is available.


In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name="scit_support_kb_day3",
)

print("Vector store created.")


Vector store created.


## 🧪 6. Test Retrieval Before Using an LLM (Grounded First)

> ⚠️ **Golden Rule**: Never start with answer generation. First verify: **Did we retrieve the right knowledge?**

This separates retrieval quality issues from LLM generation issues 🔍.

In [9]:
def retrieve(query, k=4):
    return vectorstore.similarity_search(query, k=k)

query = "كم رسوم الشحن السريع في السعودية؟"

docs = retrieve(query)

for i, doc in enumerate(docs, 1):
    print(f"\n===== Result {i} =====")
    print(doc.page_content[:700])
    print("Metadata:", doc.metadata)



===== Result 1 =====
Question: كم رسوم الشحن السريع في السعودية؟
Answer: رسوم الشحن السريع داخل المملكة العربية السعودية هي 45 ريال سعودي، ويستغرق التوصيل من يوم إلى يومين عمل في المدن الرئيسية مثل الرياض وجدة.
Keywords: رسوم, الشحن السريع, السعودية, 45 ريال, يومين
Metadata: {'source_type': 'faq', 'source_id': 'FAQ-001', 'target_country': 'KSA', 'category': 'shipping', 'updated_at': '2026-08-01'}

===== Result 2 =====
Question: متى أحصل على شحن مجاني؟
Answer: تحصل على شحن عادي مجاني داخل السعودية والإمارات إذا كانت قيمة طلبك 200 ريال/درهم أو أكثر.
Keywords: شحن مجاني, 200 ريال, السعودية, الإمارات
Metadata: {'category': 'shipping', 'source_type': 'faq', 'updated_at': '2026-08-01', 'source_id': 'FAQ-006', 'target_country': 'GCC'}

===== Result 3 =====
Question: هل تتوفر خدمة الدفع عند الاستلام وما هي رسومها؟
Answer: نعم، خدمة الدفع عند الاستلام متوفرة في السعودية والإمارات برسوم إضافية تبلغ 15 ريال/درهم تدفع لشركة التوصيل.
Keywords: الدفع عند الاستلام, COD, رسوم, 15 ريال
Metadata: {'tar

## 🔍 7. Inspect Retrieval Quality Across GCC Queries

Our evaluation set contains **20 realistic test queries**:
- 🇸🇦 **Standard Arabic**: Formal questions
- 🗣️ **Informal Gulf Dialect**: Real customer phrasing (e.g., *"كم ياخذ التوصيل للرياض؟"*, *"ابا ارجع القطعه شسوي بالظبط"*)
- ⚠️ **Typos & Dialects**: Edge case queries
- ❓ **Unsupported Questions**: Out-of-scope policies (e.g., *"هل يمكنني الحصول على خصم 50%؟"*).

In [10]:
eval_df = pd.DataFrame(eval_queries)

display(
    eval_df[
        ["test_id", "query_type", "query", "expected_source_id", "expected_chunk_keyword"]
    ].head(10)
)


,test_id,query_type,query,expected_source_id,expected_chunk_keyword
0,TEST-01,standard,كم رسوم الشحن السريع في السعودية؟,FAQ-001,45
1,TEST-02,gulf_dialect,كم ياخذ التوصيل للرياض؟,FAQ-002,الرياض
2,TEST-03,standard,كيف يمكنني إرجاع منتج قمت بشرائه؟,FAQ-003,بوليصة
3,TEST-04,gulf_dialect,كيف أرجع الشغلة اللي اشتريتها؟,FAQ-003,إرجاع
4,TEST-05,gulf_dialect,ابا ارجع القطعه شسوي بالظبط,FAQ-003,إرجاع
5,TEST-06,gulf_dialect,اذا رجعت الغرض اللي دفعته بتقسيط تمارا بيوقفون...,FAQ-004,تمارا
6,TEST-07,standard,هل تلغى أقساط تابي عند إرجاع الطلب؟,FAQ-004,تابي
7,TEST-08,standard,كيف أقوم بإلغاء طلبي؟,FAQ-005,إلغاء
8,TEST-09,gulf_dialect,اقدر اكنسل الطلب الحين؟,FAQ-005,إلغاء
9,TEST-10,standard,ما هو الحد الأدنى للحصول على شحن مجاني؟,FAQ-006,200


## 📊 8. Retrieval Evaluation (Source & Keyword Accuracy)

We calculate transparent, deterministic metrics first:
1. 🎯 **Source Retrieval Accuracy**: Does top-$k$ contain the expected `source_id`?
2. 🔑 **Keyword Hit Rate**: Does top-$k$ contain the expected critical keyword?

In [11]:
def evaluate_retrieval(test_rows, k=4):
    results = []

    for row in test_rows:
        docs = retrieve(row["query"], k=k)

        source_ids = [doc.metadata.get("source_id") for doc in docs]
        keyword = row["expected_chunk_keyword"]

        keyword_found = any(
            keyword.lower() in doc.page_content.lower()
            for doc in docs
        )

        source_found = row["expected_source_id"] in source_ids

        results.append({
            "test_id": row["test_id"],
            "query_type": row["query_type"],
            "query": row["query"],
            "expected_source_id": row["expected_source_id"],
            "retrieved_source_ids": source_ids,
            "source_found": source_found,
            "keyword_found": keyword_found,
        })

    return pd.DataFrame(results)

retrieval_results = evaluate_retrieval(eval_queries, k=4)

retrieval_results.head()


,test_id,query_type,query,expected_source_id,retrieved_source_ids,source_found,keyword_found
0,TEST-01,standard,كم رسوم الشحن السريع في السعودية؟,FAQ-001,"[FAQ-001, FAQ-006, FAQ-010, FAQ-002]",True,True
1,TEST-02,gulf_dialect,كم ياخذ التوصيل للرياض؟,FAQ-002,"[FAQ-002, FAQ-001, FAQ-010, FAQ-006]",True,True
2,TEST-03,standard,كيف يمكنني إرجاع منتج قمت بشرائه؟,FAQ-003,"[FAQ-003, FAQ-008, COMPANY_POLICY, FAQ-013]",True,True
3,TEST-04,gulf_dialect,كيف أرجع الشغلة اللي اشتريتها؟,FAQ-003,"[FAQ-003, COMPANY_POLICY, FAQ-013, FAQ-007]",True,True
4,TEST-05,gulf_dialect,ابا ارجع القطعه شسوي بالظبط,FAQ-003,"[FAQ-004, FAQ-008, FAQ-003, COMPANY_POLICY]",True,True


In [12]:
retrieval_source_accuracy = retrieval_results["source_found"].mean()
retrieval_keyword_accuracy = retrieval_results["keyword_found"].mean()

print(f"Source retrieval accuracy: {retrieval_source_accuracy:.2%}")
print(f"Keyword retrieval accuracy: {retrieval_keyword_accuracy:.2%}")


Source retrieval accuracy: 85.00%
Keyword retrieval accuracy: 85.00%


## ❌ 9. Find & Analyze Failed Retrievals

A single percentage score is not enough — we must inspect failure modes:
- 🗣️ Did the query use local slang or colloquial terms?
- ✏️ Was there a severe typo?
- 🧩 Was the chunk size too large or too small?
- 🏷️ Was the document properly tagged and indexed?

In [13]:
failed = retrieval_results[~retrieval_results["source_found"]]

print("Failed retrievals:", len(failed))

display(
    failed[
        ["test_id", "query_type", "query", "expected_source_id", "retrieved_source_ids"]
    ]
)


Failed retrievals: 3


,test_id,query_type,query,expected_source_id,retrieved_source_ids
11,TEST-12,gulf_dialect,وصلني غرض مكسور ومخروش وش الحل؟,FAQ-008,"[FAQ-007, FAQ-013, FAQ-005, FAQ-003]"
15,TEST-16,standard,هل يمكنني إرجاع عطر تم فتح غلافه؟,FAQ-012,"[FAQ-003, FAQ-008, FAQ-004, COMPANY_POLICY]"
19,TEST-20,out_of_scope,هل يمكنني الحصول على خصم 50%؟,COMPANY_POLICY,"[FAQ-003, FAQ-006, FAQ-005, FAQ-008]"


## 🧠 10. Build the RAG Answer Node (Grounded Generation)

Now we connect the LLM (`gpt-4o-mini`) to the retrieved context.

> 🔒 **Grounded Generation Rule**: The model is strictly instructed to answer **ONLY** from the provided context and never invent facts or policies.

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


In [15]:
def build_context(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source_id')}]\n{doc.page_content}"
        for doc in docs
    )


def answer_question(query, k=4):
    docs = retrieve(query, k=k)
    context = build_context(docs)

    system_prompt = f"""
You are a customer support assistant for a GCC e-commerce store.

Answer the customer's question using only the retrieved company knowledge below.

Rules:
- Do not invent policy details.
- If the answer is not supported by the context, say that the available knowledge does not provide the answer.
- Keep the answer clear and direct.
- Use the same language as the customer.
- Mention important limits, fees, time windows or conditions when they are present.

Retrieved company knowledge:
{context}
"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=query),
    ])

    return response, docs


## 💬 11. Test the Complete RAG Pipeline

Let's test our complete pipeline with a standard inquiry and inspect the grounded answer and source citations 📑:

In [16]:
response, docs = answer_question("كم رسوم الشحن السريع في السعودية؟")

print(response.content)

print("\nSources used:")
for doc in docs:
    print("-", doc.metadata.get("source_id"))


رسوم الشحن السريع داخل المملكة العربية السعودية هي 45 ريال سعودي، ويستغرق التوصيل من يوم إلى يومين عمل في المدن الرئيسية مثل الرياض وجدة.

Sources used:
- FAQ-001
- FAQ-006
- FAQ-010
- FAQ-002


## 🇸🇦 12. Test Informal Gulf Queries (Saudi & GCC Dialects)

A production assistant in the GCC must understand local phrasing seamlessly:
- *"كم ياخذ التوصيل للرياض؟"* (Delivery duration)
- *"كيف أرجع الشغلة اللي اشتريتها؟"* (Return item)
- *"ابا ارجع القطعه شسوي بالظبط"* (Return procedure)
- *"الكرتون وصلني مكسور شسوي الحين"* (Damaged delivery)

In [17]:
test_questions = [
    "كم ياخذ التوصيل للرياض؟",
    "كيف أرجع الشغلة اللي اشتريتها؟",
    "ابا ارجع القطعه شسوي بالظبط",
    "الكرتون وصلني مكسور شسوي الحين",
]

for question in test_questions:
    response, docs = answer_question(question)

    print("=" * 80)
    print("QUESTION:", question)
    print("ANSWER:", response.content)
    print("SOURCES:", [d.metadata.get("source_id") for d in docs])


QUESTION: كم ياخذ التوصيل للرياض؟
ANSWER: التوصيل العادي إلى الرياض يستغرق من 3 إلى 5 أيام عمل، بينما التوصيل السريع يستغرق من يوم إلى يومين عمل فقط.
SOURCES: ['FAQ-002', 'FAQ-001', 'FAQ-010', 'FAQ-006']
QUESTION: كيف أرجع الشغلة اللي اشتريتها؟
ANSWER: يمكنك طلب الإرجاع عبر تطبيقنا أو موقعنا من صفحة 'طلباتي' خلال 14 يوماً من الاستلام. سنصدر بوليصة إرجاع مجانية ويقوم مندوب شركة الشحن باستلام الشحنة من موقعك خلال 48-72 ساعة. تأكد من أن المنتج في حالته الأصلية وفي عبوته وتغليفه الأصلي.
SOURCES: ['FAQ-003', 'COMPANY_POLICY', 'FAQ-013', 'FAQ-007']
QUESTION: ابا ارجع القطعه شسوي بالظبط
ANSWER: يمكنك طلب الإرجاع عبر تطبيقنا أو موقعنا من صفحة "طلباتي" خلال 14 يوماً من الاستلام. سنصدر لك بوليصة إرجاع مجانية، وسيقوم مندوب شركة الشحن باستلام الشحنة من موقعك خلال 48-72 ساعة.
SOURCES: ['FAQ-004', 'FAQ-008', 'FAQ-003', 'COMPANY_POLICY']
QUESTION: الكرتون وصلني مكسور شسوي الحين
ANSWER: المعذرة، المعرفة المتاحة لا توفر إجابة عن كيفية التعامل مع الكرتون المكسور. يمكنك التواصل مع خدمة العملاء للحصول على

## ⚖️ 13. Retrieval Evaluation vs. Generation Evaluation

There are two fundamentally separate evaluation questions:

| Dimension | Question | Failure Risk |
| :--- | :--- | :--- |
| **🔍 Retrieval** | *Did we find the right documents?* | Missed context, irrelevant noise |
| **🧠 Generation** | *Did the LLM answer truthfully based on the context?* | Hallucination, ungrounded claims |

```text
Case 1: Good Retrieval + Bad Answer  ──► Prompting or LLM reasoning issue
Case 2: Bad Retrieval + Fluent Answer ──► Dangerous hallucination!
```

## 🎯 14. Ragas Framework Evaluation (Faithfulness & Relevance)

In production, we use **Ragas** to measure:
- 🛡️ **Faithfulness**: Is every claim grounded in the retrieved context? (Target: $\ge 0.85$)
- 🎯 **Answer Relevancy**: Does the answer directly address the user's question?
- 📌 **Context Precision**: Are the relevant chunks ranked high?

In [18]:
import ragas
print("Ragas version:", getattr(ragas, "__version__", "unknown"))


d:\course\practs for course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ragas version: 0.4.3


## 📂 15. Prepare a Golden Dataset

A **Golden Dataset** pairs customer questions with verified ground-truth answers and source IDs. It serves as our continuous regression testing benchmark 🧪.

In [19]:
# Generate answers for the evaluation set.
answer_rows = []

for row in eval_queries:
    response, docs = answer_question(row["query"])

    answer_rows.append({
        "test_id": row["test_id"],
        "query": row["query"],
        "answer": response.content,
        "expected_source_id": row["expected_source_id"],
        "retrieved_source_ids": [
            d.metadata.get("source_id") for d in docs
        ],
    })

answers_df = pd.DataFrame(answer_rows)

display(answers_df.head())


,test_id,query,answer,expected_source_id,retrieved_source_ids
0,TEST-01,كم رسوم الشحن السريع في السعودية؟,رسوم الشحن السريع داخل المملكة العربية السعودي...,FAQ-001,"[FAQ-001, FAQ-006, FAQ-010, FAQ-002]"
1,TEST-02,كم ياخذ التوصيل للرياض؟,التوصيل العادي إلى الرياض يستغرق من 3 إلى 5 أي...,FAQ-002,"[FAQ-002, FAQ-001, FAQ-010, FAQ-006]"
2,TEST-03,كيف يمكنني إرجاع منتج قمت بشرائه؟,يمكنك طلب الإرجاع عبر تطبيقنا أو موقعنا من صفح...,FAQ-003,"[FAQ-003, FAQ-008, COMPANY_POLICY, FAQ-013]"
3,TEST-04,كيف أرجع الشغلة اللي اشتريتها؟,يمكنك طلب الإرجاع عبر تطبيقنا أو موقعنا من صفح...,FAQ-003,"[FAQ-003, COMPANY_POLICY, FAQ-013, FAQ-007]"
4,TEST-05,ابا ارجع القطعه شسوي بالظبط,يمكنك طلب الإرجاع عبر تطبيقنا أو موقعنا من صفح...,FAQ-003,"[FAQ-004, FAQ-008, FAQ-003, COMPANY_POLICY]"


## 📈 16. Simple Reliability & Performance Report

Let's summarize our pipeline performance broken down by query type (`standard`, `gulf_dialect`, `out_of_scope`):

In [20]:
report_by_type = (
    retrieval_results
    .groupby("query_type")
    .agg(
        source_accuracy=("source_found", "mean"),
        keyword_accuracy=("keyword_found", "mean"),
        count=("test_id", "count"),
    )
    .reset_index()
)

report_by_type["source_accuracy"] = report_by_type["source_accuracy"].round(3)
report_by_type["keyword_accuracy"] = report_by_type["keyword_accuracy"].round(3)

display(report_by_type)


,query_type,source_accuracy,keyword_accuracy,count
0,gulf_dialect,0.875,0.875,8
1,out_of_scope,0.000,0.000,1
2,standard,0.909,0.909,11


## 🔄 17. Exercise 1 — Impact of Hyperparameter $k$ ($k=2, 4, 6$)

Test how the number of retrieved chunks affects source accuracy and token efficiency.

> 💡 **Observation**: Higher $k$ increases recall but adds noise and token costs!

In [ ]:
for k in [2, 4, 6]:
    result = evaluate_retrieval(eval_queries, k=k)
    accuracy = result["source_found"].mean()

    print(f"k={k} -> source retrieval accuracy: {accuracy:.2%}")


## 🛡️ 18. Exercise 2 — Test an Unsupported Question (Anti-Hallucination)

Ask something that is **NOT** covered by the policies (e.g., *"هل يمكنني الحصول على خصم 50%؟"*).

The model should politely decline rather than fabricating a discount code 🚫.

In [ ]:
unsupported_question = "هل يمكنني الحصول على خصم 50%؟"

response, docs = answer_question(unsupported_question)

print("QUESTION:", unsupported_question)
print("ANSWER:", response.content)
print("SOURCES:", [d.metadata.get("source_id") for d in docs])


## 🔍 19. Exercise 3 — Inspect Sources & Citations

Inspect the exact chunks and metadata retrieved for complex questions like Tamara BNPL installment cancellation 💳.

In [ ]:
question = "اذا رجعت الغرض اللي دفعته بتقسيط تمارا بيوقفون الاقساط الباقية؟"

response, docs = answer_question(question)

print("ANSWER")
print(response.content)

print("\nRETRIEVED CONTEXT")
for i, doc in enumerate(docs, 1):
    print(f"\n--- SOURCE {i}: {doc.metadata.get('source_id')} ---")
    print(doc.page_content)


## 🏆 20. Day 3 Assignment & Submission Guide

Complete and test your RAG pipeline end-to-end:

```text
Company Policies & FAQs ──► Chunking ──► Embeddings ──► ChromaDB ──► Top-k Retriever ──► LLM ──► Grounded Answer & Evaluation
```

### 📋 Submission Checklist:
1. 📸 Screenshot of a successful retrieval & grounded answer.
2. 📊 Calculated Source Retrieval Accuracy percentage.
3. 🛠️ Analysis of 1 failed query and your adjustment.
4. ⚙️ Your optimal $k$ value justification.

## 🎓 21. Key Takeaways Before Day 4

Before moving to **Day 4 (FastAPI, Streaming & Production Deployment)**, verify your understanding:

1. 💡 **RAG Architecture**: Why knowledge injection beats fine-tuning for dynamic policies.
2. ✂️ **Chunking & Overlap**: Balancing semantic coherence with context size.
3. 🔢 **Vector Embeddings & ChromaDB**: Geometric similarity search.
4. ⚖️ **Two-Stage Evaluation**: Evaluating Retrieval separately from Generation.
5. 🛡️ **Faithfulness & Guardrails**: Preventing hallucinations and unsupported promises.